# Lesson 1: Agentic RAG - Code Along 1

Following the lessons and experimenting with different models.
The course teaches how to use OpenAI, but I also want to try Anthropic.

## Preparations

In [ ]:
# Dependencies

# basic functionalities
import os
import requests
from dotenv import load_dotenv

# search
from minsearch import Index

# model providers
# for this tutorial, I experiment with both OpenAI and Anthropic
from openai import OpenAI
from anthropic import Anthropic


In [2]:
# load environment variables (Anthropic API key)
load_dotenv()

True

## Anthropic

In [3]:
# intitialise Anthropic client
# uses API key from .env file
anthropic_client = Anthropic()

In [4]:
# get any arbitrary message to test this
# use Haiku, because it's the smallest and cheapest model
anthropic_test_response = anthropic_client.messages.create(
    # select latest Haiku 3.5 model
    model="claude-haiku-4-5",
    # limit number of tokens to 50 to keep cost low, this is just for testing
    max_tokens=50,
    # use a simple user message to test this
    messages=[
        {
            "role": "user",
            "content": "Hey how's it going?"
        }
    ]
)

# print message part of response
print(anthropic_test_response.content[0].text)

Hey! Doing well, thanks for asking. Just here and ready to chat or help with whatever you need. What's on your mind?


## OpenAI

In [5]:
# I use Anthropic as model provider, but I can still use OpenAI SDK
# this works using a workaround by providing a URL to Anthropic
client = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url="https://api.anthropic.com/v1/"
)

In [6]:
# use an arbitrary message to test this
# use Haiku, because it's the smallest and cheapest model
openai_test_response = client.chat.completions.create(
    model="claude-haiku-4-5",
    # limit number of tokens to 50 to keep cost low, this is just for testing
    max_tokens=50,
    # use a simple user message to test this
    messages=[
        {"role": "user", "content": "Hey how's it going?"}
    ]
)

# print message part of response
print(openai_test_response.choices[0].message.content)

Hey! Doing well, thanks for asking. Just here and ready to chat or help with whatever you need. What's up with you?


## Get the Course Q&A Docs Dataset for RAG

In [7]:
# URL where to find the Q&A docs
# this contains a list of courses
docs_url = "https://datatalks.club/faq/json/courses.json"

# get the docs
docs = requests.get(docs_url)

# access the docs as JSON
courses_raw = docs.json()

# print the content to have a look at it
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 83},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [8]:
# fetch all actual FAQ documents for all courses

# initialize list containing Q&A pairs and set URL prefix for access
documents = []
url_prefix = "https://datatalks.club/faq"

# iterate over all courses and fetch data
for course in courses_raw:
    # construct full URL
    course_url = f"""{url_prefix}{course["path"]}"""

    # fetch data
    course_response = requests.get(course_url)
    
    # raise error if response is not OK
    course_response.raise_for_status()

    # access data in JSON format
    course_data = course_response.json()

    # extend list with data
    documents.extend(course_data)

# print number of Q&A pairs
print(len(documents))

# print first few Q&A pairs to get an impression
for doc in documents[:5]:
    print(doc)

1346
{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}
{'id': 'bfafa427b3', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: What are the prerequisites for this course?', 'answer': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not requir

## Index Documents for Search

In [9]:
# use minsearch for indexing the Q&A docs

# define how to index
index = Index(
    # tokenize and search these as text
    text_fields=["question", "section", "answer"],
    # use for literal filtering
    keyword_fields=["course"]
)

# fit the index
index.fit(documents)

In [12]:
# try a search
index.search(
    # actual question
    query="What is this course about?",
    # filter for course - in this case for the current one - LLM Zoomcamp
    filter_dict={"course": "llm-zoomcamp"},
    # apply boosts to make certain fields more influential, some less so
    boost_dict={"question": 2, "section": 0.5},
    # limit number of results
    num_results=5
)


[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/l

In [22]:
# turn this into a function
def search(question, course="llm-zoomcamp", num_results=5):
    
    filter_dict={"course": course}
    boost_dict={"question": 2, "section": 0.5}
    
    # try a search
    return index.search(
        query=question,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=num_results,
    )

In [28]:
search("Does this course teach about agents?")

[{'id': '96286b4be4',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Why does the content keep changing?',
  'answer': 'If the homework title contains `[DRAFT]`, it means the homework is not ready yet.\n\nThe homework is ready only when both are true:\n\n- The homework form is open on the course management platform.\n- The homework title does not contain `[DRAFT]`.\n\nUntil then, the content can still change. Working on the material or homework in advance is at your own risk, because the final version can be different.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](ht

## Building the Prompt

In [34]:
# system prompt -> static
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

# user prompt -> variable
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [36]:
# define function for building the context (the formatted search results)
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [45]:
# try out the function (limited to 3 search results for better printing)

# save search results to object for reusability
search_results = index.search(
    # actual question
    query="What is this course about?",
    # filter for course - in this case for the current one - LLM Zoomcamp
    filter_dict={"course": "llm-zoomcamp"},
    # apply boosts to make certain fields more influential, some less so
    boost_dict={"question": 2, "section": 0.5},
    # limit number of results
    num_results=3
)

print(build_context(search_results=search_results))

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

Module 1: RAG
Q: OpenAI: Do I have to subscribe and pay for Open AI API for this course?
A: No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.

See the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives).

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoo

In [47]:
# build user prompt by combining question and context

# define function for doing this
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

# save question to object for reusability
question = "I just discovered the course. Can I join now?"

# call the function on the question and the context (search results)
# still using just first 3 search results for better printing
prompt = build_prompt(
    question=question,
    search_results=search_results,
)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

Module 1: RAG
Q: OpenAI: Do I have to subscribe and pay for Open AI API for this course?
A: No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.

See the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives).

General Course-Related Questions
Q: How should I start the

In [52]:
# define function for calling an LLM
# pass instructions as system prompt and user prompt containing question and context
# I decided to use Anthrpic and translate the course's OpenAI calls to it
# use small model for prototyping and limit number of tokens
def llm(instructions, user_prompt, model="claude-haiku-4-5", max_tokens=1024):
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=instructions,
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.content[0].text

In [57]:
# call RAG
# model and max tokens are set to default
llm_response = llm(
    instructions=INSTRUCTIONS,
    user_prompt=prompt,
)

In [58]:
print(llm_response)

Yes, you can join the course now! 

According to the course information, you can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

Here's how to get started:

1. **Check the documentation**: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

2. **Follow the typical workflow**:
   - Watch the lesson videos
   - Work through the lesson notebooks/code
   - Read the homework instructions on GitHub
   - Submit your answers through the course platform before the deadline

One important thing to note: if you want to earn a certificate, you'll need to finish the course with a "live" cohort, as certificates are only awarded for the live co

In [60]:
# now wire it all up into one single function
def rag(query, model="claude-haiku-4-5"):
    # search
    search_results = search(query)
    # prompt
    prompt = build_prompt(query, search_results)
    # llm
    answer = llm(INSTRUCTIONS, prompt, model)
    
    return answer

In [65]:
# try it out
print(rag("Is there a Slack channel? If so, how to join?"))

Yes, there is a Slack channel for the course. According to the context, you should:

1. **Use the course Slack channel** for technical discussion and questions
2. **Join via the announcements** - The video URL and links for live sessions are posted in the announcements channel on both Telegram and Slack

The context mentions that if you're stuck on something, you should ask your question in Slack and follow the [asking questions guidelines](https://datatalks.club/docs/courses/zoomcamp-logistics/asking-questions/).

However, the specific link or invitation details for joining the Slack channel are not provided in the context. You may need to check the course homepage or initial enrollment materials for the direct Slack workspace link.
